# Chapter 08: Model Comparison as Code Efficiency
### Cross-Entropy, WAIC & PSIS-LOO

> *"Everything should be made as simple as possible, but not simpler."*
> — Albert Einstein

In Chapter 07, we saw how hierarchical modeling compresses test fleet flakiness,
balancing individual noise against fleet-wide stability via the Information
Bottleneck.

Now suppose your machine learning or reliability engineering team builds three
competing statistical models to predict whether a pull request build will fail
due to test flakiness:
1. **Model 1 (Flat Bernoulli)**: A single global flakiness rate
   $\theta_{\text{fleet}}$.
2. **Model 2 (Hierarchical by Service)**: Partial pooling across microservices
   with
   random intercepts.
3. **Model 3 (Massive Feature Interaction Model)**: 45 covariates including
   commit
   author, lines of code changed, time of day, CI runner host ID, git branch
   name, and CPU cache misses.

On historical training data, Model 3 achieves an astonishing 99.4% accuracy,
predicting past test outcomes almost without flaw. Model 1 achieves 81%, and
Model 2 achieves 89%.

Should you deploy Model 3 to your production CI infrastructure?

If you do, your CI budget will likely explode and your false positive rate will
surge. Model 3 has memorized the idiosyncratic noise of past runs—it is
catastrophically overfitted.

How do we scientifically compare models and penalize complexity without
guessing?
- In **Information Theory**, the answer is **Minimum Description Length (MDL)
  and
  Cross-Entropy**: the best model compresses unseen out-of-sample test results
  into the shortest binary code.
- In **Bayesian Statistics**, the answer is **Out-of-Sample Predictive Density
  via
  WAIC and PSIS-LOO**: estimating generalization error while using the variance
  of posterior surprise to diagnose overconfidence.

This chapter unifies Bayesian model comparison with information-theoretic
compression.



## 1. Out-of-Sample Predictive Density as Compression

When Claude Shannon established information theory in 1948, he proved that the
optimal code length to transmit a sequence of symbols drawn from true
distribution $p(y)$ using model distribution $q(y)$ is the cross-entropy:

$$
H(p, q) = -\mathbb{E}_{p}[\log_2 q(y)] = H(p) + D_{\text{KL}}(p \parallel q)
$$

In Bayesian statistics, when we evaluate a model $\mathcal{M}$ parameterized by
$\boldsymbol{\theta}$ with posterior $p(\boldsymbol{\theta} \mid
\mathcal{D}_{\text{train}})$, the predictive distribution for a new, unseen
build outcome $\tilde{y}$ is:

$$
p(\tilde{y} \mid \mathcal{D}_{\text{train}}) = \int p(\tilde{y} \mid \boldsymbol{\theta}) p(\boldsymbol{\theta} \mid \mathcal{D}_{\text{train}}) d\boldsymbol{\theta}
$$

The **Expected Log Pointwise Predictive Density (elpd)** measures the average
log probability our model assigns to unseen out-of-sample data points
$\tilde{y}_i$:

$$
\text{elpd} = \sum_{i=1}^n \mathbb{E}_{f}[\ln p(\tilde{y}_i \mid \mathcal{D}_{\text{train}})]
$$

Notice the exact identity connecting predictive density to information theory:

$$
\text{elpd} = - n \ln(2) \cdot H(f, p_{\text{post}})
$$

**Maximizing out-of-sample Bayesian predictive accuracy is mathematically
identical to minimizing out-of-sample transmission code length.** A model that
overfits to training data allocates code words to historical noise, leaving
long, inefficient code words for future builds.

### Marginal Likelihood, Evidence, and Bayes Factors ($BF_{10}$)
While $\text{elpd}$ evaluates predictive density pointwise across data points,
the Bayesian **Marginal Likelihood (Evidence)** evaluates the total probability
of observing the entire dataset $D$ under model $\mathcal{M}$:

$$
p(D \mid \mathcal{M}) = \int p(D \mid \boldsymbol{\theta}, \mathcal{M}) \, p(\boldsymbol{\theta} \mid \mathcal{M}) \, d\boldsymbol{\theta}
$$

Under Shannon coding, the marginal evidence has an immediate physical
interpretation: **$-\log_2 p(D \mid \mathcal{M})$ is the exact description
length in bits needed to compress the observed test history $D$ under model
$\mathcal{M}$**.

When comparing two competing models $\mathcal{M}_0$ (baseline) and
$\mathcal{M}_1$ (degraded / alternative), their ratio of marginal evidences
defines the **Bayes Factor**:

$$
BF_{10} = \frac{p(D \mid \mathcal{M}_1)}{p(D \mid \mathcal{M}_0)}
$$

Taking the base-2 logarithm gives the **weight of evidence in bits**:

$$
\Delta L = \log_2 BF_{10} = \left[-\log_2 p(D \mid \mathcal{M}_0)\right] - \left[-\log_2 p(D \mid \mathcal{M}_1)\right]
$$

A positive $\log_2 BF_{10}$ indicates how many bits shorter the test data's
description is under $\mathcal{M}_1$ compared to $\mathcal{M}_0$.

### Concrete Beta-Binomial Example: Healthy Baseline vs. Degraded Flake
Suppose a critical CI test fails $k = 3$ times in $n = 20$ runs. We compare:
- **$\mathcal{M}_0$ (Healthy Baseline):** The test is healthy with fixed rate
$\theta = 0.005$ ($0.5\%$ background noise).
- **$\mathcal{M}_1$ (Degraded Flaky Model):** The test has degraded into a
flaky state with prior belief $\theta \sim \text{Beta}(1, 19)$ (prior mean
$5.0\%$).

Under $\mathcal{M}_0$, the evidence is the single binomial point likelihood:

$$
p(D \mid \mathcal{M}_0) = \binom{20}{3} (0.005)^3 (0.995)^{17} \approx 0.000131
$$

Under $\mathcal{M}_1$, integrating over $\text{Beta}(1, 19)$ yields the analytic
Beta-Binomial marginal evidence:

$$
p(D \mid \mathcal{M}_1) = \binom{20}{3} \frac{B(1 + 3, 19 + 17)}{B(1, 19)} = 1140 \cdot \frac{B(4, 36)}{B(1, 19)} \approx 0.06055
$$

The resulting Bayes Factor is:

$$
BF_{10} = \frac{p(D \mid \mathcal{M}_1)}{p(D \mid \mathcal{M}_0)} \approx \frac{0.06055}{0.000131} \approx \mathbf{462.5}
$$

In information units, this represents:

$$
\log_2 BF_{10} = \log_2(462.5) \approx \mathbf{+8.85\text{ bits}}
$$

An information surplus of $+8.85\text{ bits}$ constitutes overwhelming evidence
(well beyond Harold Jeffreys' threshold of $\log_2(100) \approx 6.64$ bits for
'decisive' evidence) that the test endpoint has drifted into a degraded flaky
state.



## 2. WAIC: Penalizing the Variance of Surprise

Calculating true out-of-sample predictive density requires infinite unseen test
runs. In practice, we only have our training set of $n$ observed test results
$y_1, \dots, y_n$.

If we evaluate log-predictive density on the training data directly, we get the
**Log Pointwise Predictive Density (lppd)**:

$$
\text{lppd} = \sum_{i=1}^n \ln \left( \frac{1}{S} \sum_{s=1}^S p(y_i \mid
\boldsymbol{\theta}^{(s)}) \right)
$$

where $S$ is the number of MCMC posterior draws.

The training $\text{lppd}$ is always overly optimistic because the parameters
were tuned to fit those exact $y_i$. How much optimism penalty must we subtract?

Akihiro Watanabe (2010) derived the **Widely Applicable Information Criterion
(WAIC)**, which computes the effective number of parameters $p_{\text{WAIC}}$
directly from the posterior distribution:

$$
p_{\text{WAIC}} = \sum_{i=1}^n \text{Var}_{\text{post}}\left(\ln p(y_i \mid
\boldsymbol{\theta})\right)
$$


Look closely at this formula through the lens of information theory:
- $\ln p(y_i \mid \boldsymbol{\theta})$ is the **self-information (surprise)**
  of
  datum $y_i$ under parameter draw $\boldsymbol{\theta}$.
- $\text{Var}_{\text{post}}(\ln p(y_i \mid \boldsymbol{\theta}))$ is the
  **variance of surprise** across the posterior distribution!

If a parameter vector is tightly constrained by the data, every posterior sample
$\boldsymbol{\theta}^{(s)}$ assigns approximately the same probability to $y_i$,
so the variance of surprise is near zero. If a model is overly complex and
flexible, different posterior draws disagree wildly on how to explain $y_i$,
causing the variance of surprise to explode!

The WAIC metric on deviance scale (multiplying by $-2$) is:

$$
\text{WAIC} = -2 \left( \text{lppd} - p_{\text{WAIC}} \right)
$$


Lower WAIC indicates shorter expected code length on unseen future builds.



## 3. PSIS-LOO: Exact Leave-One-Out Cross-Validation Without Re-Fitting

The gold standard for out-of-sample validation is **Leave-One-Out Cross-
Validation (LOO-CV)**: fit the model on $n-1$ data points, and compute the log-
likelihood of the single held-out point $y_i$, repeating for all $i=1, \dots,
n$:

$$
\text{elpd}_{\text{loo}} = \sum_{i=1}^n \ln p(y_i \mid \mathcal{D}_{-i})
$$


In production, running full MCMC $n$ separate times (where $n = 50,000$ test
builds) would cost thousands of GPU hours.

### Importance Sampling to the Rescue
We can estimate $p(y_i \mid \mathcal{D}_{-i})$ using posterior draws from the
full-data model $p(\boldsymbol{\theta} \mid \mathcal{D})$ by reweighting with
importance weights:

$$
r_i^{(s)} = \frac{p(\boldsymbol{\theta}^{(s)} \mid
\mathcal{D}_{-i})}{p(\boldsymbol{\theta}^{(s)} \mid \mathcal{D})} \propto
\frac{1}{p(y_i \mid \boldsymbol{\theta}^{(s)})}
$$


The importance weight is simply the inverse likelihood of datum $y_i$! If a
datum is highly surprising to the model ($p(y_i \mid \boldsymbol{\theta})$ is
tiny), its importance weight $r_i$ explodes to infinity, causing importance
sampling variance to become infinite.

### Pareto Smoothing (Vehtari, Gelman, Gabry 2017)
To stabilize heavy-tailed importance ratios, Vehtari et al. fit a **Generalized
Pareto Distribution (GPD)** to the upper $20\%$ largest weights $r_i^{(s)}$:

$$
F(r) = 1 - \left(1 + \frac{k(r - \mu)}{\sigma}\right)^{-1/k}
$$


The shape parameter $k$ is the famous **Pareto $k$ diagnostic**:
- **$k < 0.5$**: The importance distribution has finite variance; LOO estimates
  converge rapidly.
- **$0.5 \le k < 0.7$**: Slower convergence, but still practically reliable.
- **$k \ge 0.7$**: The distribution has infinite variance! The importance
  weights
  are unstable; datum $y_i$ is an extreme high-leverage outlier that exerts
  excessive pull on the posterior.

```
       Pareto k Diagnostic Gauge:
       [==== Safe (k < 0.5) ====][== Caution (0.5 <= k < 0.7) ==][==== DANGER (k
>= 0.7) ====]
       Finite Variance          Slow Convergence                 Infinite
Variance (Outlier)
       LOO-CV Highly Reliable   Acceptable                       Datum exerts
excessive leverage
```



## 4. Minimum Description Length and Bayesian Model Stacking

When comparing multiple candidate models $\mathcal{M}_1, \mathcal{M}_2, \dots,
\mathcal{M}_M$, we compute their difference in predictive performance:

$$
\Delta \text{elpd}_{12} = \text{elpd}_{\text{loo}}^{(1)} - \text{elpd}_{\text{loo}}^{(2)}
$$

along with the standard error of the difference:

$$
\text{SE}(\Delta \text{elpd}) = \sqrt{n \cdot \text{Var}_i \left( \text{elpd}_{\text{loo}, i}^{(1)} - \text{elpd}_{\text{loo}, i}^{(2)} \right)}
$$

A model is definitively superior only if:

$$
\Delta \text{elpd} > 2 \times \text{SE}(\Delta \text{elpd})
$$

### Model Stacking (Optimal Ensemble Compression)
Rather than picking a single 'winning' model (which discards complementary
information), we can combine all $M$ models with weights $\mathbf{w} = (w_1,
\dots, w_M)^T$ by solving a strictly concave log-score maximization:

$$
\max_{\mathbf{w} \in \Delta^M} \sum_{i=1}^n \ln \left( \sum_{m=1}^M w_m p(y_i \mid \mathcal{D}_{-i}, \mathcal{M}_m) \right)
$$

This is equivalent to constructing a **multi-code mixture compressor** that
achieves optimal cross-entropy by hedging across models.

### Bayesian Two-Sample Comparison: 'Did My CL Actually Fix the Flake?'
In reliability engineering, developers do not merely compare static models; they
submit deflaking changelists (CLs) and need to answer a vital operational
question: **Did my commit actually reduce test flakiness, or was the clean re-
run just lucky?**

Consider comparing the failure rate before the fix (Baseline $A$) with the
failure rate after the fix (Candidate $B$):
- **Baseline $A$ (Before CL):** Observed $k_A = 8$ failures in $n_A = 50$
runs. With an uninformative $\text{Beta}(1, 1)$ prior, the posterior is

$$
\theta_A \sim \text{Beta}(1 + 8, 1 + 42) = \text{Beta}(9, 43)
$$

with posterior mean $\mathbb{E}[\theta_A] = \frac{9}{52} \approx 17.3\%$.
- **Candidate $B$ (After CL):** Observed $k_B = 0$ failures in $n_B = 30$
verification runs. With a $\text{Beta}(1, 1)$ prior, the posterior is

$$
\theta_B \sim \text{Beta}(1 + 0, 1 + 30) = \text{Beta}(1, 31)
$$

with posterior mean $\mathbb{E}[\theta_B] = \frac{1}{32} \approx 3.1\%$.

Rather than computing an uncalibrated frequentist $p$-value, Bayesian decision
theory computes the exact posterior **Probability of Improvement**:

$$
P(\theta_B < \theta_A \mid D_A, D_B) = \int_0^1 P(\theta_B < \theta \mid D_B) \, p(\theta_A = \theta \mid D_A) \, d\theta \approx \mathbf{98.84\%}
$$

Furthermore, we can compute the probability of a substantial, practical
improvement—for instance, that the commit reduced flakiness by at least a factor
of $2\times$ ($50\%$ reduction):

$$
P(\theta_B < 0.5\,\theta_A \mid D_A, D_B) \approx \mathbf{94.3\%}
$$

This gives engineering leads an explicit, calibrated confidence metric: there is
a $98.8\%$ chance the patch improved the test, and a $94.3\%$ chance the defect
rate was more than halved.



## 5. What Do We Actually Do With This Number?
### From Wasted Bits to Operational Action

Every theoretical diagnostic in this chapter maps directly to software and CI
decisions.

### Action 1: Automated Gating for ML/CI Models
- **The Problem**: A data scientist submits a PR adding a new
Transformer-based flakiness predictor. It claims $99\%$ training accuracy.
- **The Operational Rule**: Compute $\Delta \text{elpd}_{\text{loo}}$ against
the production baseline.
  - If $\Delta \text{elpd}_{\text{loo}} \le 2 \times \text{SE}$, **reject the
    PR**.
The model's complexity is not justified by out-of-sample data compression.
  - If $\Delta \text{elpd}_{\text{loo}} > 2 \times \text{SE}$, **approve for
    shadow
deployment**.

### Action 2: Triaging Outliers via Pareto $k \ge 0.7$
- **The Problem**: CI pipelines occasionally experience catastrophic anomalous
failures (e.g., Jenkins worker ran out of disk space, or network switch dropped
packets). These 'black swans' poison regular flaky test models.
- **The Operational Rule**: Filter the dataset using the Pareto diagnostic:
  - If a failed test run $y_i$ has Pareto $k_i \ge 0.7$, **it is an
    environmental
outlier, not code flakiness**.
  - Automatically tag the run as an *Infrastructure Outlier*, isolate it from
test flakiness scorecards, and route the incident to SRE/DevOps rather than the
developer who authored the commit.

### Action 3: Gating Deflake CLs via Posterior Probability of Improvement
- **The Problem**: A developer claims their CL fixed a flaky test by running
it 10 times locally without failure. Flaky tests routinely pass 10 times in a
row by chance.
- **The Operational Rule**: Require $P(\theta_B < \theta_A \mid D) \ge 95\%$
and $P(\theta_B < 0.5\,\theta_A \mid D) \ge 90\%$ before removing a test from
quarantine. As computed above, observing $0$ failures in $30$ runs against a
baseline of $8/50$ satisfies both bounds ($98.8\%$ and $94.3\%$), safely
certifying the fix.

### Action 4: Log Compression and Cloud Compute Sizing
- **The Problem**: Storing raw CI execution traces for 100,000 builds per day
costs thousands of dollars per month in S3/BigQuery.
- **The Operational Rule**: Convert cross-entropy differences $\Delta H$
directly to storage dollars:

$$
\text{Storage Savings (GB/year)} = \frac{\Delta H \text{ (bits/build)} \times
100,000 \text{ builds/day} \times 365}{8 \times 10^9}
$$

A model that achieves 0.8 bits/build cross-entropy reduction over naive
Bernoulli encoding saves gigabytes of log payload daily when compressing test
telemetry.



## 6. Summary and Roadmap

| Concept | Information Theory View | Bayesian Statistics View | Operational CI Action |
| :--- | :--- | :--- | :--- | 
| **Out-of-sample accuracy** | Minimal cross-entropy $H(p, q)$ code length | Maximal $\text{elpd}_{\text{loo}}$ predictive density | Predict build failure on future PRs |
| **Model complexity penalty** | Description length of parameters $L(\mathcal{M})$ | Variance of surprise $p_{\text{WAIC}} = \sum \text{Var}(\ln p)$ | Reject overparameterized 45-feature models |
| **Bayes Factor ($BF_{10}$)** | Description length savings $\Delta L = -\log_2 BF_{10}$ | Ratio of marginal evidences $p(D \mid \mathcal{M}_1) / p(D \mid \mathcal{M}_0)$ | Confirm test drift ($+8.85\text{ bits}$ flake evidence) |
| **Deflake A/B Testing** | Relative code efficiency between versions | $P(\theta_B < \theta_A \mid D)$ posterior mass | Verify fix before lifting quarantine ($98.8\%$) |
| **Pareto $k$ diagnostic** | Code word length variance / infinite moment | Posterior leverage / outlier influence | Route infrastructure crashes away from devs |
| **Model Stacking** | Mixture codebook with optimal bit allocation | Cross-validation weight optimization | Combine static analysis with dynamic flakiness |

In the next chapter (**Chapter 09: To Merge or Re-Run? — Bayesian Decisions &
Wald SPRT**), we will turn these predictive models into active operational
decisions: *Given our current posterior and asymmetric risk costs, should we
merge the PR, block the build, or run one more test retry?*

